# Ingesta

## Imports

In [45]:
import requests
import pandas as pd
import sqlite3
import os


## Transformación + Carga en Raw 

In [46]:
# Configuración de variables globales y conexión a DB

In [47]:
URL = "https://world.openbeautyfacts.org/cgi/search.pl"
NUM_BATCHES = 10
TOTAL_PAGES = 1000
PAGES_PER_BATCH = TOTAL_PAGES//NUM_BATCHES
DB_NAME = "openbeauty"
os.makedirs("data", exist_ok=True)
DB_PATH = f"data/{DB_NAME}.db"

TABLE_NAME = "raw_products"

conn = sqlite3.connect(DB_PATH)


In [ ]:
# Datos seleccionados del JSON para usare para el proyecto

In [23]:
def extract_columns(product_dict):
    keys = [
        "_id",
        "code",
        "rev",
        "update_key",
    
        "brands",
        "product_name",
        "product_type",
    
        "countries",
        "countries_tags",
        "countries_hierarchy",
    
        "categories_hierarchy",
        "categories",
    
        "product_quantity",
        "product_quantity_unit",
        "quantity",
    
        "ingredients_n",
        "known_ingredients_n",
        "unknown_ingredients_n",
    
        "completeness",
    
        "scans_n",
        "unique_scans_n",
        "popularity_tags",
    
        "created_t",
        "last_modified_t",
        "last_updated_t",
    
        "creator",
        "last_editor",
        "last_modified_by",
    
        "data_quality_tags"
    ]
    filtered_data = [{k: item.get(k) for k in keys} for item in product_dict]
    return filtered_data

In [52]:
# Función para hacer la petición get a la api, retornado la lista batch_data del numero de paginas correspondiente al batch
# El lote se hará desde initial_page hasta end_page

In [51]:
def load_batch_data(initial_page, end_page):
    batch_data = []

    for n_page in range(initial_page, end_page):
        params = {"search_terms":"","page":n_page, "json":1}
        response = requests.get(URL, params=params)

        if response.status_code == 200:
            data = response.json()
            filtered_data = extract_columns(data['products'])
            # Reconstruye la lista de diccionarios agregando "page":n_page, para control
            filtered_data = [dict(item, **{"page":n_page}) for item in filtered_data] 
            batch_data = batch_data + filtered_data
        
    return batch_data
        

In [ ]:
# Función para insertar en base de datos en crudo (raw) los datos resultantes de la función load_batch_data(initial_page, end_page)

In [50]:
def insert_raw_data(batch_data, batch, conn):
    try:
        product_df = pd.DataFrame(batch_data)
        product_df["batch_id"] = batch + 1
        product_df['dtinserted'] = datetime.datetime.now()
        product_df = product_df.map(lambda x: json.dumps(x) if isinstance(x, (list, dict)) else x)
        product_df.to_sql(TABLE_NAME, conn, if_exists="append", index=False)

        print("Data inserted")
        
    except Exception as e:
        print(f"Exception occurs: {e}")
    
        